1. Expense Calculator Agent (Cyclic Graph – Loops) 

In [ ]:
import os
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

# 1. Define Tools
@tool
def add_expenses(a: float, b: float) -> float:
    """Adds two expense amounts together."""
    return a + b

@tool
def apply_tax(amount: float, tax_percent: float) -> float:
    """Applies a percentage tax (like GST) to an amount."""
    return amount * (1 + (tax_percent / 100))

@tool
def split_bill(amount: float, people: int) -> float:
    """Splits an amount equally among a specific number of people."""
    if people <= 0:
        raise ValueError("Number of people must be greater than zero.")
    return amount / people

tools = [add_expenses, apply_tax, split_bill]
tool_node = ToolNode(tools)

# 2. Define State
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# 3. Define the Agent Node
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools)

def call_model(state: AgentState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

# 4. Define Routing Logic
def should_continue(state: AgentState) -> Literal["tools", "__end__"]:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return "__end__"

# 5. Build the Cyclic Graph
workflow = StateGraph(AgentState)

workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue)
workflow.add_edge("tools", "agent") # This closes the loop

app = workflow.compile()

# Example Invocation
# inputs = {"messages": [("user", "Add 2000 and 1500, apply 18% tax, and split among 3 people.")]}
# for chunk in app.stream(inputs, stream_mode="values"):
#     chunk["messages"][-1].pretty_print()

2. Travel Planner (Memory-Based Bot) 


In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

class PlannerState(TypedDict):
    messages: Annotated[list, add_messages]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

def assistant(state: PlannerState):
    sys_msg = SystemMessage(content=(
        "You are a helpful travel assistant. Remember the user's destination, "
        "budget/luxury preferences, and travel dates as they share them over time. "
        "When asked to summarize, provide a clean digest of their choices."
    ))
    response = llm.invoke([sys_msg] + state["messages"])
    return {"messages": [response]}

# Build Graph with Memory
builder = StateGraph(PlannerState)
builder.add_node("assistant", assistant)
builder.add_edge(START, "assistant")

# MemorySaver enables multi-turn conversation persistence
memory = MemorySaver()
travel_bot = builder.compile(checkpointer=memory)

# Simulation of Multi-Turn Conversation
config = {"configurable": {"thread_id": "user_session_abc123"}}

# Turn 1
# travel_bot.invoke({"messages": [("user", "I am planning a trip to Bali.")]}, config)
# Turn 2
# travel_bot.invoke({"messages": [("user", "I prefer budget hotels.")]}, config)
# Turn 3
# travel_bot.invoke({"messages": [("user", "My trip is in December.")]}, config)
# Turn 4 (Summary Request)
# response = travel_bot.invoke({"messages": [("user", "Summarise my travel plan.")]}, config)
# print(response["messages"][-1].content)

 3. Access Control Approval System (Human-in-the-Loop) 


In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

# 1. Admin Tools
@tool
def grant_access(user: str, system: str) -> str:
    """Grants administrative or user access to a specific corporate IT system."""
    return f"SUCCESS: Granted {system} access to user '{user}'."

@tool
def revoke_access(user: str, system: str) -> str:
    """Revokes access from a specific corporate IT system."""
    return f"SUCCESS: Revoked access from user '{user}' for system {system}."

tools = [grant_access, revoke_access]
tool_node = ToolNode(tools)

class AdminState(TypedDict):
    messages: Annotated[list, add_messages]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools)

def call_admin_agent(state: AdminState):
    return {"messages": [llm.invoke(state["messages"])]}

def route_after_agent(state: AdminState) -> Literal["tools", "__end__"]:
    if state["messages"][-1].tool_calls:
        return "tools"
    return "__end__"

# 2. Wire Up Architecture
admin_builder = StateGraph(AdminState)
admin_builder.add_node("agent", call_admin_agent)
admin_builder.add_node("tools", tool_node)

admin_builder.add_edge(START, "agent")
admin_builder.add_conditional_edges("agent", route_after_agent)
admin_builder.add_edge("tools", "agent")

# Compile with an explicit interrupt barrier before tools execute
memory_buffer = MemorySaver()
admin_system = admin_builder.compile(
    checkpointer=memory_buffer,
    interrupt_before=["tools"]
)

# --- Execution Lifecycle Scenario ---
config = {"configurable": {"thread_id": "request_789"}}
initial_input = {"messages": [("user", "Give database admin access to user 'Kiran'")]}

# Step 1: Run until the pause point
# event = admin_system.invoke(initial_input, config)
# (Graph execution yields here. It stops right before 'tools'.)

# Step 2: Read current state to see what tool the agent wants to call
# state_snapshot = admin_system.get_state(config)
# proposed_tool_calls = state_snapshot.values["messages"][-1].tool_calls
# print("Pending Authorization for:", proposed_tool_calls)

# Step 3: Human interaction simulation
# user_approval = input("Approve action? (yes/no): ")

# if user_approval.strip().lower() == "yes":
    # Resume execution passing None to pick up exactly where it was frozen
#     admin_system.invoke(None, config)
# else:
#     print("Action explicitly rejected by administrator.")